This Colab notebook includes all the code you need to experiment with page-level document classification using a RAG-based strategy. Upload your own multi-document pharmaceutical PDF, follow along step-by-step, and learn how to detect document boundaries, label each section by type, and generate clean metadata for smarter extraction and automation.


**Step 1: Extract Page-Level Content from Pharmaceutical PDF**





In [ ]:
# ============================================
# STEP 1: Install Required Libraries
# ============================================
#
# WHAT WE'RE DOING:
# Installing PyPDF2, a Python library for reading and extracting
# text from PDF files.
#
# WHY THIS MATTERS:
# Pharmaceutical bundled files are multi-document PDFs. We need a
# reliable way to extract text from each page before we can
# classify or route documents.
#
# WHAT YOU'LL SEE:
# A brief installation log confirming PyPDF2 was installed.
# ============================================

!pip install -q PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.9 MB/s eta 0:00:00


In [ ]:
# ============================================
# STEP 2: Extract Page-Level Content from PDF
# ============================================
#
# WHAT WE'RE DOING:
# Loading the pharmaceutical blob PDF and extracting the raw
# text from every page into a list of dictionaries.
#
# WHY THIS MATTERS:
# Each page in a pharma blob file may belong to a different
# document (Certificate of Quality, BSE/TSE Declaration, etc.).
# Extracting per-page text is the foundation for classification.
#
# WHAT YOU'LL SEE:
# A list of dictionaries, each containing a page number and
# the extracted text content for that page.
# ============================================

from PyPDF2 import PdfReader

reader = PdfReader("/content/pharma-blob-sample.pdf")
pages = [page.extract_text() for page in reader.pages]
doc_pages = [{"page_num": i, "text": p} for i, p in enumerate(pages)]
doc_pages

[{'page_num': 0,
  'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752\nUnited States\nPage 1 / 1\ncytiva.com\n3 June, 2022\nRe: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating\nInstructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits\nas well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to\nbrittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to\n+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of\ndamage to the kit during setup and handling.\nDescription Part Number Operating Temperature\nHigh Flow Kit F, Modified, AKTA ready 29477427 +2°C to +40°C\nHigh Flow Gradient C, Modified, AKTA ready 29184612 +2°C to +4

**Step 2: Set Up Gemini LLM Connection**

In [ ]:
# ============================================
# STEP 3: Set Up Gemini LLM Connection
# ============================================
#
# WHAT WE'RE DOING:
# Creating a helper function that sends a prompt to Google's
# Gemini 2.0 Flash model and returns the response text.
#
# WHY THIS MATTERS:
# We use an LLM to classify pharmaceutical document types and
# detect document boundaries -- tasks that require understanding
# natural language content on each page.
#
# WHAT YOU'LL SEE:
# No output -- this cell defines the function for later use.
# ============================================

def gemini_model(prompt):
    import google.generativeai as genai
    from google.colab import userdata

    genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

    model = genai.GenerativeModel("models/gemini-2.0-flash")
    response = model.generate_content(prompt)

    return response.text

In [ ]:
# ============================================
# STEP 4: Write Document Boundary Detection Function
# ============================================
#
# WHAT WE'RE DOING:
# Building a function that asks the LLM whether two consecutive
# pages belong to the same pharmaceutical document.
#
# WHY THIS MATTERS:
# Pharma blob files bundle multiple documents back-to-back.
# Detecting where one document ends and another begins is the
# critical first step before classification or extraction.
#
# WHAT YOU'LL SEE:
# True or False -- whether pages 1 and 2 (index 1 and 2) belong
# to the same document given the type "Certificate of Quality".
# ============================================

def is_same_document(prev_text, curr_text, doc_type=None):
    prompt = f"""
    You are checking whether two consecutive pages from a pharmaceutical
    document package belong to the SAME individual document.

    A NEW document starts when the page has:
    - A different document title or heading (e.g., "Certificate of Quality"
      vs "Packaging Specification" vs "Material Description Sheet")
    - A completely different topic or subject matter
    - Its own header with a new document number or reference

    Pages belong to the SAME document when:
    - The second page says "continued" or "page 2 of 2"
    - The content directly continues the previous page's discussion
    - They share the same document number or title

    Previous page type: {doc_type or 'unknown'}

    Previous Page:
    {prev_text}

    Current Page:
    {curr_text}

    Do these two pages belong to the SAME document? Answer ONLY 'Yes' or 'No'.
    """
    response = gemini_model(prompt)
    return response.strip().lower().startswith("yes")


prev_text = doc_pages[1]["text"]
curr_text = doc_pages[2]["text"]
doc_type = "Certificate of Quality"  # Optional, can be "unknown" or None

is_same_document(prev_text, curr_text, doc_type)

False

**Step 3: Write the Document Type Classifier**

In [ ]:
# ============================================
# STEP 5: Write Document Type Classifier
# ============================================
#
# WHAT WE'RE DOING:
# Building a function that classifies the first page of each
# new document into a pharmaceutical document category.
#
# WHY THIS MATTERS:
# Once we detect a boundary, we need to label the new document.
# Accurate classification enables downstream routing -- e.g.,
# sending a Certificate of Quality to QA and a BSE/TSE
# Declaration to regulatory review.
#
# WHAT YOU'LL SEE:
# The predicted document type for a sample page.
# ============================================

def classify_document_type(text):
    prompt = f"""
    You are a pharmaceutical document classifier. Based on the page
    content below, classify it into ONE of these document types:

    - Cover Letter: A formal letter (often starting with "To Whom It
      May Concern") discussing product information or storage conditions.
    - Certificate of Quality: Contains lot numbers, manufacture dates,
      expiration dates, and test results (autoclave, gamma irradiation).
    - Packaging Specification: Describes packaging components, materials,
      part numbers, and configuration change history.
    - BSE/TSE Declaration: A declaration about animal-origin materials
      and transmissible spongiform encephalopathy compliance.
    - Material Description: Lists materials of construction, sterilization
      compatibility, and physical properties of a product.
    - Supplier Qualification: Contains supplier audit history,
      certifications (ISO 9001, ISO 13485), and approved product lists.
    - Chain of Custody: Lists manufactured assemblies, traceability
      information, and the manufacturing-to-shipment flow.
    - Other: Use ONLY if the content does not match any of the above.

    Page Content:
    {text}

    Respond with ONLY the document type name. No explanation.
    """
    response = gemini_model(prompt).strip().lower().replace(".", "")
    result = response.title()
    return result

classify_document_type(doc_pages[3]["text"])

'Packaging Specification'

**Step 4: Loop Through Pages and Generate Page-Level Metadata**

In [ ]:
# ============================================
# STEP 6: Loop Through Pages and Generate Metadata
# ============================================
#
# WHAT WE'RE DOING:
# Iterating through every page in the PDF. For each page we
# check if it continues the previous document or starts a new
# one, then record its document ID and type.
#
# WHY THIS MATTERS:
# This is the core pipeline -- it produces the page-level
# metadata table that tells us exactly which pages belong to
# which pharmaceutical document and what type each document is.
#
# WHAT YOU'LL SEE:
# A printed list of dictionaries showing page number, document
# ID, and document type for every page in the blob file.
# ============================================

results = []
current_doc_type = None
doc_counter = 0

for i, page in enumerate(doc_pages):
    if i == 0:
        current_doc_type = classify_document_type(page["text"])
    else:
        prev_text = doc_pages[i - 1]["text"]
        same = is_same_document(prev_text, page["text"], current_doc_type)
        if not same:
            doc_counter += 1
            current_doc_type = classify_document_type(page["text"])

    results.append({
        "page": i,
        "doc_id": doc_counter,
        "doc_type": current_doc_type
    })


for r in results:
    print(r)

{'page': 0, 'doc_id': 0, 'doc_type': 'Cover Letter'}
{'page': 1, 'doc_id': 1, 'doc_type': 'Certificate Of Quality'}
{'page': 2, 'doc_id': 2, 'doc_type': 'Certificate Of Quality'}
{'page': 3, 'doc_id': 3, 'doc_type': 'Packaging Specification'}
{'page': 4, 'doc_id': 3, 'doc_type': 'Packaging Specification'}
{'page': 5, 'doc_id': 4, 'doc_type': 'Bse/Tse Declaration'}
{'page': 6, 'doc_id': 5, 'doc_type': 'Material Description'}
{'page': 7, 'doc_id': 6, 'doc_type': 'Supplier Qualification'}
{'page': 8, 'doc_id': 6, 'doc_type': 'Supplier Qualification'}
{'page': 9, 'doc_id': 7, 'doc_type': 'Chain Of Custody'}


**Step 5: Visualize Results**

In [ ]:
# ============================================
# STEP 7: Visualize Results
# ============================================
#
# WHAT WE'RE DOING:
# Loading the metadata into a pandas DataFrame for a clean,
# tabular view of the classification results.
#
# WHY THIS MATTERS:
# A DataFrame makes it easy to filter, sort, and export the
# results -- for example, pulling all pages that belong to
# "Certificate of Quality" documents for QA review.
#
# WHAT YOU'LL SEE:
# A table with columns: page, doc_id, doc_type showing the
# classification for every page in the pharmaceutical blob file.
# ============================================

import pandas as pd

df = pd.DataFrame(results)
df

,page,doc_id,doc_type
0,0,0,Cover Letter
1,1,1,Certificate Of Quality
2,2,2,Certificate Of Quality
3,3,3,Packaging Specification
4,4,3,Packaging Specification
5,5,4,Bse/Tse Declaration
6,6,5,Material Description
7,7,6,Supplier Qualification
8,8,6,Supplier Qualification
9,9,7,Chain Of Custody
